# End-to-End Multi-Agent Research Test

Full pipeline test with harness-driven iterative research: upload a document, perform deep research, and receive a comprehensive analytical report.

## 1. Load Environment

In [1]:
import os
import json
import time
import httpx
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)
print("✅ Environment loaded")

✅ Environment loaded


## 2. Verify All Agents Running

In [2]:
agents = {
    "orchestrator": "http://localhost:8100",
    "doc_processor": "http://localhost:8101",
    "researcher": "http://localhost:8102",
    "writer": "http://localhost:8103",
    "reviewer": "http://localhost:8104",
}

all_ready = True
for name, url in agents.items():
    try:
        resp = httpx.get(f"{url}/.well-known/agent-card.json", timeout=5)
        if resp.status_code == 200:
            print(f"  ✅ {name}")
        else:
            print(f"  ❌ {name}: HTTP {resp.status_code}")
            all_ready = False
    except Exception:
        print(f"  ❌ {name}: not reachable")
        all_ready = False

if all_ready:
    print("\n✅ All agents ready")
else:
    print("\n❌ Some agents not running. Run: make agents-start")

  ❌ orchestrator: not reachable
  ❌ doc_processor: not reachable
  ❌ researcher: not reachable
  ❌ writer: not reachable
  ❌ reviewer: not reachable

❌ Some agents not running. Run: make agents-start


## 3. Upload and Ingest a Document

First, ingest a document through the Document Processor agent.

In [ ]:
sample_doc = os.path.join(os.path.dirname(os.getcwd()), "data", "sample_docs", "sample.pdf")

if not os.path.exists(sample_doc):
    import urllib.request
    os.makedirs(os.path.dirname(sample_doc), exist_ok=True)
    urllib.request.urlretrieve("https://arxiv.org/pdf/2408.09869", sample_doc)
    print(f"Downloaded sample document")

# Send to doc_processor via A2A
payload = {
    "jsonrpc": "2.0",
    "method": "message/send",
    "id": "ingest-001",
    "params": {
        "message": {
            "role": "user",
            "parts": [{"kind": "text", "text": sample_doc}],
        }
    },
}

print(f"Ingesting: {os.path.basename(sample_doc)}")
start = time.time()

try:
    resp = httpx.post("http://localhost:8101", json=payload, timeout=300)
    result = resp.json()
    elapsed = time.time() - start
    print(f"Response: {json.dumps(result, indent=2)[:500]}")
    print(f"\nIngestion took {elapsed:.1f}s")
    print("✅ Document ingested")
except Exception as e:
    print(f"❌ Ingestion failed: {e}")

## 4. Perform Deep Research via Orchestrator

Send a research query through the full multi-agent pipeline.

In [ ]:
research_query = (
    "Provide a comprehensive analysis of the document processing techniques, "
    "architecture decisions, and key innovations described in the paper. "
    "Include specific technical details and their implications."
)

payload = {
    "jsonrpc": "2.0",
    "method": "message/send",
    "id": "research-001",
    "params": {
        "message": {
            "role": "user",
            "parts": [{"kind": "text", "text": research_query}],
        }
    },
}

print(f"Research query: {research_query[:80]}...")
print("\nExecuting multi-agent pipeline: plan → research → write → review")
print("(This may take 30-60 seconds)\n")

start = time.time()

try:
    resp = httpx.post("http://localhost:8100", json=payload, timeout=300)
    result = resp.json()
    elapsed = time.time() - start

    # Extract final report
    report = ""
    if "result" in result:
        for artifact in result["result"].get("artifacts", []):
            for part in artifact.get("parts", []):
                if part.get("kind") == "text":
                    report += part["text"]
        if not report and "message" in result["result"]:
            for part in result["result"]["message"].get("parts", []):
                if part.get("kind") == "text":
                    report += part["text"]

    print("=" * 60)
    print("RESEARCH REPORT")
    print("=" * 60)
    print(report if report else json.dumps(result, indent=2)[:2000])
    print(f"\n{'='*60}")
    print(f"Pipeline completed in {elapsed:.1f}s")
    print(f"Report length: {len(report)} characters")
    print("✅ End-to-end research pipeline successful")

except httpx.TimeoutException:
    print(f"❌ Request timed out after 300s")
except Exception as e:
    print(f"❌ Error: {e}")

## 5. Performance Summary

In [ ]:
print("Custom Deep Research — Performance Summary")
print("=" * 50)
print(f"Agents used:        5 (orchestrator + 4 workers)")
print(f"Protocol:           A2A (JSON-RPC 2.0 over HTTP)")
print(f"Agent framework:    LangGraph (StateGraph)")
print(f"Agent platform:     Kagenti ADK")
print(f"Document parser:    Docling")
print(f"Vector store:       PostgreSQL + pgvector")
print(f"Model serving:      RHOAI vLLM")
print()
print("Pipeline steps:")
print("  1. Plan research strategy (Orchestrator)")
print("  2. Ingest document (Doc Processor → Docling → pgvector)")
print("  3. Search & synthesize (Researcher → pgvector RAG)")
print("  4. Generate report (Writer)")
print("  5. Quality review (Reviewer)")
print("  6. Finalize output (Orchestrator)")
print()
print("✅ Lab Phase 4 complete")